In [ ]:
!pip install -U transformers
!pip install tiktoken sentencepiece

In [ ]:
!pip install tiktoken sentencepiece

In [ ]:
#from kaggle_secrets import UserSecretsClient
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
my_token = user_secrets.get_secret("HF_TOKEN")

In [ ]:
import os
os.environ['HF_TOKEN'] = my_token

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import asyncio
import uuid
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
topics = [
    "Quantum Mechanics",
    "Artificial Intelligence",
    "Roman Empire",
    "Abstract Expressionism",
    "Marine Biology",
    "Microeconomics",
    "Space Exploration",
    "Cognitive Psychology",
    "Climate Change",
    "Renewable Energy",
    "Cyber Security",
    "Classical Music",
    "Cryptography",
    "World War II",
    "Culinary Arts",
    "Botany",
    "Astrophysics",
    "Political Science",
    "Evolutionary Biology",
    "Renaissance Art",
    "Data Science",
    "Stoicism",
    "Organic Chemistry",
    "Ancient Egypt",
    "Photography",
    "Game Theory",
    "Nanotechnology",
    "Linguistics",
    "Mythology",
    "Public Health",
    "Architecture",
    "Genetic Engineering",
    "Sociology",
    "Jazz History",
    "Robotics",
    "Criminology",
    "Urban Planning",
    "Blockchain",
    "Epidemiology",
    "Poetry",
    "Neuroscience",
    "Existentialism",
    "Oceanography",
    "Graphic Design",
    "Geology",
    "Machine Learning",
    "Medieval History",
    "Cinema Studies",
    "Zoology",
    "Calculus",
    "Thermodynamics",
    "Software Engineering",
    "International Relations",
    "Fashion Design",
    "Astronomy",
    "Anthropology",
    "Digital Marketing",
    "Philosophy of Mind",
    "Paleontology",
    "Virtual Reality",
    "Typography",
    "Particle Physics",
    "Ancient Greece",
    "Behavioral Economics",
    "Biotechnology",
    "Forestry",
    "Supply Chain Management",
    "Calligraphy",
    "Anatomy",
    "Ethics",
    "Number Theory",
    "Human Rights",
    "Interior Design",
    "Meteorology",
    "3D Printing",
    "French Revolution",
    "Music Theory",
    "Agronomy",
    "Immunology",
    "Social Psychology",
    "E-commerce",
    "Quantum Computing",
    "Industrial Design",
    "Ornithology",
    "Game Development",
    "Archaeology",
    "Epistemology",
    "Real Estate",
    "Ceramics",
    "Fluid Dynamics",
    "Biomechanics",
    "Modernism",
    "Virology",
    "Information Theory",
    "Astrobiology",
    "Demography",
    "Cartography",
    "Aerospace Engineering",
    "Folk Music",
    "Mycology"
]

In [ ]:
prompt = """
# ROLE:
You are an expert in the field of formal logic and Syllogism creation in natural language.
# WORKFLOW:
Your goal is to Generate a syllogism in a predefined format {formats} based on the topic of {topic} so it is {validity} and {plausibility}.
#Step 1:
Generate syllogism in format of 2 premises and 1 conclusion based on properties described above. The output should look something like this
```There are people who are students. A few children are students. It follows, then, that some children are people.```
# NOTES:
- Do NOT Include any special symbols like '⊢'.
- Make sure that there are always 3 sentences.
- Please, return only the syllogism in the specified format.
"""

In [ ]:
formats = [
"Aab,Abc ⊢ Aac meaning: All b are a, All c are b therefore All c are a. E.g. All humans  are mortal , All Greeks  are humans . Therefore, all Greeks  are mortal .",

"Eab,Abc ⊢ Eac meaning: No b are a, All c are b therefore No c are a. E.g. No reptiles  have fur , All snakes  are reptiles . Therefore, no snakes  have fur .",

"Aab,Ibc ⊢ Iac meaning: All b are a, Some c are b therefore Some c are a. E.g. All birds  have feathers , Some pets  are birds . Therefore, some pets have feathers .",

"Eab,Ibc ⊢ Oac meaning: No b are a, Some c are b therefore Some c are not a. E.g. No vegetarians  eat meat , Some athletes  are vegetarians. Therefore, some athletes do not eat meat."
]

In [ ]:
model_path = "/kaggle/input/models/janflajk/qwen/pytorch/default/1/models/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28"


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False,trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True
)



In [ ]:
async def run_inference(topics,formats,prompt:str ,enable_thinking:bool=True,max_topics:int=100):
    ans = []
    max_topics = min(len(topics), max_topics)
    
    for topic in tqdm(topics[:max_topics]):
        for f in formats:
            for p in [True, False]:
                for v in [True, False]:
                    try:
                        validity = "not valid" if not v else "valid"
                        plausibility = "not plausible" if not p else "plausible"


                        messages = [
                            {"role": "user", "content": prompt.format(
                                formats = f,
                                topic = topic,
                                validity = validity,
                                plausibility = plausibility
                            )
                            }
                        ]
                        text = tokenizer.apply_chat_template(
                            messages,
                            tokenize=False,
                            add_generation_prompt=True,
                            enable_thinking=enable_thinking,
                            )
                        
                        model_inputs = tokenizer([text], return_tensors="pt").to("cuda")
                        generated_ids = model.generate(
                            **model_inputs,
                            max_new_tokens=64,
                            do_sample=True,
                            temperature=0.9,
                            pad_token_id=tokenizer.eos_token_id
                        )
                        
                        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
                        response = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
                        
            
                        ans.append({"id": uuid.uuid4(),"syllogism":response.replace("\n", " "), "validity": v, "plausible": p})
                    except Exception as e:
                        print(e)
                        ans.append("network_or_connectivity_issue")
            
    return ans

In [ ]:
inference = await run_inference(topics,formats,prompt,enable_thinking=True,max_topics=100)

In [ ]:
df = pd.DataFrame(inference)

In [ ]:
df["syllogism"].nunique()

In [ ]:
df.to_csv("dataset_final.csv",index=False)

In [ ]:
df_cleaned = df.drop_duplicates(subset=['syllogism'], keep=False)
df_cleaned.to_json("data_task1.json",orient='records')